# Kannada ↔ Hindi Code-Switching Translator — Interactive Notebook

Type your **own** Kannada text (romanized like `naanu mane hogutthini`, or native script like `ನಾನು ಮನೆ ಹೋಗುತ್ತೇನೆ`), mixed with Hindi, and get a Hindi translation — with a color-coded breakdown of which words were detected as Kannada / Hindi / English, a visualization of the language mix, a switch between the offline dictionary and a real neural MT model, and a way to teach the system new words on the fly.

**Run the cells in order** (Cell → Run All is easiest). The interactive widgets need `ipywidgets`, installed in the first cell.


In [1]:
%pip install -q scikit-learn indic-transliteration ipywidgets matplotlib pandas
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
Dependencies ready.


## 1. Load the pipeline

This imports the core modules shipped alongside this notebook (`lexicon.py`, `lang_id.py`, `transliterator.py`, `translator.py`, `pipeline.py`). Make sure this notebook is in the **same folder** as those files.


In [2]:
import sys, os
sys.path.insert(0, os.getcwd())

from pipeline import CodeSwitchTranslator
from translator import DictionaryBackend, normalize_to_kannada_script

translator = CodeSwitchTranslator(verbose=False)
print(f"Pipeline loaded. Active backend: {translator.backend.name}")
print(f"Dictionary size: {len(translator.backend.kn_dict)} Kannada entries")

Pipeline loaded. Active backend: dictionary-fallback
Dictionary size: 231 Kannada entries


## 2. Interactive Translator — type your own sentence

Type Kannada (romanized *or* native ಕನ್ನಡ), Hindi, or a mix of both (English loanwords are fine too), then click **Translate**.

- Each word is shown color-coded by detected language, with its tag underneath.
- Below that, the Hindi translation.
- A coverage line tells you how many Kannada words were actually found in the dictionary (vs. passed through untranslated).


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

TAG_COLORS = {
    "KN": "#f4a261",     # Kannada
    "HI": "#2a9d8f",     # Hindi
    "EN": "#8ecae6",     # English
    "NUM": "#adb5bd",    # numbers
    "PUNCT": "#e9ecef",  # punctuation
    "OTHER": "#e76f51",  # named entities
}
TAG_NAMES = {
    "KN": "Kannada", "HI": "Hindi", "EN": "English",
    "NUM": "Number", "PUNCT": "Punctuation", "OTHER": "Named entity",
}

def render_tagged_html(tagged_tokens):
    spans = []
    for tok, tag in tagged_tokens:
        if tag == "PUNCT":
            continue
        color = TAG_COLORS.get(tag, "#dee2e6")
        spans.append(
            f'''<span style="display:inline-block;background:{color};
            padding:4px 10px;margin:3px;border-radius:6px;
            font-family:sans-serif;text-align:center;">
            <div style="font-size:15px;">{tok}</div>
            <div style="font-size:9px;opacity:0.75;">{tag}</div>
            </span>'''
        )
    return '<div style="line-height:2.6;">' + " ".join(spans) + "</div>"

input_box = widgets.Textarea(
    value="Naanu appa mane hogutthini, thoda late aagutte.",
    placeholder="Type Kannada (romanized or ಕನ್ನಡ), Hindi, or mixed text here...",
    description="Input:",
    layout=widgets.Layout(width="100%", height="70px"),
    style={"description_width": "60px"},
)
translate_btn = widgets.Button(description="Translate ➔ Hindi", button_style="success",
                                icon="language")
output_area = widgets.Output()

def on_translate_click(b):
    with output_area:
        clear_output()
        text = input_box.value.strip()
        if not text:
            print("Type something first!")
            return

        tagged = translator.lid.tag_sentence(text)
        display(HTML("<b>Detected languages:</b>"))
        display(HTML(render_tagged_html(tagged)))

        result = translator.translate(text)
        display(HTML(
            f'<b>Hindi translation:</b>'
            f'<div style="font-size:22px;padding:12px;background:#f8f9fa;'
            f'border-radius:8px;margin-top:8px;border:1px solid #dee2e6;">{result}</div>'
        ))

        kn_tokens = [t for t, tag in tagged if tag == "KN"]
        if kn_tokens and hasattr(translator.backend, "kn_dict"):
            found = sum(
                1 for w in kn_tokens
                if normalize_to_kannada_script(w) in translator.backend.kn_dict
            )
            pct = 100 * found / len(kn_tokens)
            display(HTML(
                f"<i>Dictionary coverage: {found}/{len(kn_tokens)} "
                f"Kannada tokens found ({pct:.0f}%)</i>"
            ))

translate_btn.on_click(on_translate_click)
display(input_box, translate_btn, output_area)
on_translate_click(None)  # run once immediately so the notebook shows output on open

Textarea(value='Naanu appa mane hogutthini, thoda late aagutte.', description='Input:', layout=Layout(height='…

Button(button_style='success', description='Translate ➔ Hindi', icon='language', style=ButtonStyle())

Output()

## 3. Visualize the language mix of your sentence

Uses the **same text box** above — click the button below any time after editing it.


In [4]:
import matplotlib.pyplot as plt
from collections import Counter

viz_btn = widgets.Button(description="Show language mix chart", button_style="info", icon="pie-chart")
viz_output = widgets.Output()

def on_viz_click(b):
    with viz_output:
        clear_output()
        text = input_box.value.strip()
        if not text:
            print("Type something in the box in Section 2 first!")
            return
        tagged = translator.lid.tag_sentence(text)
        counts = Counter(tag for _, tag in tagged if tag != "PUNCT")
        if not counts:
            print("Nothing to show.")
            return
        labels = [f"{TAG_NAMES.get(l, l)}" for l in counts]
        sizes = list(counts.values())
        colors = [TAG_COLORS.get(l, "#dee2e6") for l in counts]

        fig, ax = plt.subplots(figsize=(4.5, 4.5))
        ax.pie(sizes, labels=labels, autopct="%1.0f%%", colors=colors, startangle=90,
               textprops={"fontsize": 10})
        ax.set_title(f"Token-level language composition\n(\"{text[:40]}{'...' if len(text) > 40 else ''}\")")
        plt.tight_layout()
        plt.show()

viz_btn.on_click(on_viz_click)
display(viz_btn, viz_output)
on_viz_click(None)

Button(button_style='info', description='Show language mix chart', icon='pie-chart', style=ButtonStyle())

Output()

## 4. Switch translation backend

- **Dictionary (offline)** — instant, no downloads, limited vocabulary (curated by hand).
- **Neural MT — IndicTrans2** — real Kannada→Hindi neural translation. Requires `transformers` + `torch` (and an internet connection to download the model the first time, a few GB). If unavailable, it automatically falls back to the dictionary.


In [5]:
backend_dropdown = widgets.Dropdown(
    options=[
        ("Dictionary (offline, instant)", "dict"),
        ("Neural MT — IndicTrans2 (needs transformers + torch)", "nmt"),
    ],
    value="dict",
    description="Backend:",
    style={"description_width": "80px"},
)
backend_status = widgets.Output()

def switch_backend(change):
    with backend_status:
        clear_output()
        if change["new"] == "nmt":
            try:
                from translator import NMTBackend
                print("Loading IndicTrans2 — this can take a while the first time "
                      "(downloading model weights)...")
                translator.backend = NMTBackend()
                print(f"✅ Switched to: {translator.backend.name}")
            except Exception as e:
                print(f"⚠️ Could not load the NMT backend ({e}).")
                print("Falling back to the dictionary backend. "
                      "Install with: pip install transformers torch sentencepiece")
                translator.backend = DictionaryBackend()
        else:
            translator.backend = DictionaryBackend()
            print(f"✅ Switched to: {translator.backend.name}")

backend_dropdown.observe(switch_backend, names="value")
display(backend_dropdown, backend_status)

Dropdown(description='Backend:', options=(('Dictionary (offline, instant)', 'dict'), ('Neural MT — IndicTrans2…

Output()

## 5. Teach it new words (live dictionary editing)

The offline dictionary only knows a curated set of words. Add your own Kannada → Hindi entries here — they take effect immediately, so you can go back up to Section 2 and try a sentence using the word you just added.

Type the Kannada word in **either** native script (ಕನ್ನಡ) or romanized form — both work.


In [6]:
from transliterator import to_kannada
from lang_id import _KANNADA_RE

new_kn = widgets.Text(description="Kannada word:", placeholder="e.g. ಸನ್ತೊಷ or santosha",
                       style={"description_width": "100px"})
new_hi = widgets.Text(description="Hindi meaning:", placeholder="e.g. खुशी",
                       style={"description_width": "100px"})
add_btn = widgets.Button(description="Add to dictionary", button_style="warning", icon="plus")
add_output = widgets.Output()

def on_add_click(b):
    with add_output:
        clear_output()
        kn_word = new_kn.value.strip()
        hi_word = new_hi.value.strip()
        if not kn_word or not hi_word:
            print("Fill in both fields first.")
            return
        if not hasattr(translator.backend, "kn_dict"):
            print("The current backend (NMT) doesn't use a dictionary — "
                  "switch to the dictionary backend in Section 4 to use this.")
            return
        kn_script = kn_word if _KANNADA_RE.match(kn_word) else to_kannada(kn_word)
        translator.backend.kn_dict[kn_script] = hi_word
        print(f"Added: {kn_script} → {hi_word}")
        print("Go back to Section 2 and try a sentence using this word!")

add_btn.on_click(on_add_click)
display(new_kn, new_hi, add_btn, add_output)

Text(value='', description='Kannada word:', placeholder='e.g. ಸನ್ತೊಷ or santosha', style=TextStyle(description…

Text(value='', description='Hindi meaning:', placeholder='e.g. खुशी', style=TextStyle(description_width='100px…

Button(button_style='warning', description='Add to dictionary', icon='plus', style=ButtonStyle())

Output()

## 6. Sample sentences

A quick batch run over a few built-in test sentences, shown as a table.


In [7]:
import pandas as pd

SAMPLES = [
    "Naanu office hogutthini, thoda late aagutte.",
    "Nanna appa mane bahut door hai.",
    "ನಾನು ಇಂದು ಮನೆ మें हूं.",
    "Naale somavaara, exam ide.",
    "Amma tumba chennagide, naanu tumba khushi aagide.",
]

rows = []
for s in SAMPLES:
    rows.append({"Input": s, "Hindi Output": translator.translate(s)})

df = pd.DataFrame(rows)
df.style.set_properties(**{"text-align": "left", "font-size": "13px"})

,Input,Hindi Output
0,"Naanu office hogutthini, thoda late aagutte.","मैं ऑफिस जा रहा हूँ, थोद late ಆಗುತ್ತೇ."
1,Nanna appa mane bahut door hai.,मेरा पिता घर बहुत् door है.
2,ನಾನು ಇಂದು ಮನೆ మें हूं.,मैं आज घरమ ें हूं.
3,"Naale somavaara, exam ide.","कल सोमवार, exam है."
4,"Amma tumba chennagide, naanu tumba khushi aagide.","माँ तुम्ब अच्छा है, मैं तुम्ब खुशि ಆಗಿದೇ."


## Notes & limitations

- The dictionary backend only knows the words in `lexicon.py` (~250 entries after Section 5 additions in this session) — anything else is passed through untranslated, in Kannada script, by design.
- Words you add in Section 5 only persist for this notebook session. To make them permanent, add them to `KANNADA_NATIVE_TO_HINDI` in `lexicon.py`.
- For production-quality translation on arbitrary text, use the NMT backend (Section 4).
- Full write-up of the architecture: see `README.md` / `report.tex` (or `report.pdf`) in this project folder.
